# Biological and matched-random nectary-search trajectories

Run in order to reproduce the analyses from the included saved replay summary. The final section displays one raw trajectory on the simulated flower.

In [ ]:
from pathlib import Path
import os
import sys

PACKAGE_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "NectarySearch").is_dir() and (p / "FlowerTracking").is_dir()),
    None,
)
if PACKAGE_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted code folder.")
NOTEBOOK_DIR = PACKAGE_ROOT / "NectarySearch"
sys.path.insert(0, str(PACKAGE_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))
os.chdir(NOTEBOOK_DIR)

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from utils.plotting import plot_confidence_intervals, plot_raincloud

In [ ]:
def filter_outliers(pd_group):
    Q1 = pd_group.quantile(0.25)
    Q3 = pd_group.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (pd_group >= lower_bound) & (pd_group <= upper_bound)

In [ ]:
deora_replay_df = pd.read_csv("results/deora_replay_summary_2030.csv")
deora_biological = deora_replay_df.loc[deora_replay_df['trajectory_type'] == 'biological', :]

In [ ]:
metric = "mean_realized_information_gain"

In [ ]:
outlier_mask = deora_biological.groupby('c_value')[metric].transform(filter_outliers)
condition_vals = deora_biological[outlier_mask].groupby('c_value')[metric]
condition_stats = condition_vals.agg(['mean', 'sem'])
condition_arrays = condition_vals.apply(np.array)

In [ ]:
plot_raincloud(*[condition_arrays[v] for v in [-1, -2, -3, -100]],
               labels=['Funnel', 'Near-funnel', 'Near-flat', 'Flat'],
               )

In [ ]:
plot_confidence_intervals(*[condition_arrays[v] for v in [-1, -2, -3, -100]],
                          labels=['Funnel', 'Near-funnel', 'Near-flat', 'Flat'],
                          alpha=0.05,
                          ylim=[0.2, 1.5])

## Biological vs Matched Random Information Gain

In [ ]:
from matplotlib.patches import Patch

condition_order = [-1, -2, -3, -100]
condition_labels = ["Funnel", "Near-funnel", "Near-flat", "Flat"]
trajectory_order = ["biological", "matched_random"]
trajectory_labels = {
    "biological": "Biological",
    "matched_random": "Matched random",
}
trajectory_colors = {
    "biological": "tab:blue",
    "matched_random": "tab:orange",
}

split_df = deora_replay_df[deora_replay_df["trajectory_type"].isin(trajectory_order)].copy()
split_outlier_mask = split_df.groupby(["c_value", "trajectory_type"])[metric].transform(filter_outliers)
split_condition_arrays = (
    split_df[split_outlier_mask]
    .groupby(["c_value", "trajectory_type"])[metric]
    .apply(lambda values: values.dropna().to_numpy())
)

fig, ax = plt.subplots(figsize=(8, 4))
box_width = 0.32
trajectory_offsets = {
    "biological": -0.5 * box_width,
    "matched_random": 0.5 * box_width,
}
box_data = []
box_positions = []
box_colors = []

for x, c_value in enumerate(condition_order, start=1):
    for trajectory_type in trajectory_order:
        values = split_condition_arrays.get((c_value, trajectory_type), np.array([]))
        if len(values) == 0:
            continue
        box_data.append(values)
        box_positions.append(x + trajectory_offsets[trajectory_type])
        box_colors.append(trajectory_colors[trajectory_type])

boxplot = ax.boxplot(
    box_data,
    positions=box_positions,
    widths=box_width,
    patch_artist=True,
    showfliers=False,
)

for patch, color in zip(boxplot["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_edgecolor(color)
    patch.set_alpha(0.45)

for median in boxplot["medians"]:
    median.set_color("black")
    median.set_linewidth(1.5)

for line_key in ["whiskers", "caps"]:
    for line, color in zip(boxplot[line_key], np.repeat(box_colors, 2)):
        line.set_color(color)

ax.axhline(0, color="0.75", linewidth=0.8, linestyle="--", zorder=0)
ax.set_xticks(range(1, len(condition_order) + 1))
ax.set_xticklabels(condition_labels)
ax.set_xlabel("Flower condition")
ax.set_ylabel("Mean realized information gain")
ax.set_title("Deora replay mean realized information gain")
ax.legend(
    handles=[
        Patch(
            facecolor=trajectory_colors[trajectory_type],
            edgecolor=trajectory_colors[trajectory_type],
            alpha=0.45,
            label=trajectory_labels[trajectory_type],
        )
        for trajectory_type in trajectory_order
    ],
    frameon=False,
)
fig.tight_layout()

## Plot Deora Trajectory on Env Canvas

This helper overlays a Deora proboscis-tip trajectory on the allocentric environment canvas. The Deora circle radius is treated as the flower radius, so normalized radius 1 maps to `env.flower.R`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = PACKAGE_ROOT

from NectarySearch.deora_data import (
    DEORA_DATA_ROOT,
    deora_actions_for_env,
    load_circle_parameters,
    load_deora_trajectory,
)
from NectarySearch.nectary_allocentric_env import AllocentricNectarSearchEnv, CanvasParams
from NectarySearch.nectary_env import FlowerParams

def plot_deora_trajectory_on_env(
    track_path,
    *,
    data_root=DEORA_DATA_ROOT,
    env=None,
    scale_mode="corolla_radius",
    max_radius_normalized=None,
    min_points=2,
    depth=1.0,
    head_standoff=0,
    canvas_padding=10.0,
    line_color="tab:red",
    ax=None,
):
    """Plot one Deora trajectory over the allocentric env canvas image."""
    data_root = Path(data_root)
    if not data_root.is_absolute():
        data_root = REPO_ROOT / data_root

    track_path = Path(track_path)
    if not track_path.is_absolute():
        track_path = REPO_ROOT / track_path

    circle_parameters = load_circle_parameters(data_root)
    trajectory = load_deora_trajectory(
        track_path,
        circle_parameters,
        data_root=data_root,
        max_radius_normalized=max_radius_normalized,
        min_points=min_points,
    )

    if env is None:
        flower = FlowerParams(c=trajectory.metadata.c_value, R=100.0, r0=5.0)
        max_r = float(np.nanmax(trajectory.radius_normalized)) * flower.R
        half_extent = max(flower.R, max_r) + float(canvas_padding)
        canvas = CanvasParams(
            width=2.0 * half_extent,
            height=2.0 * half_extent,
            pattern_type="none",
        )
        env = AllocentricNectarSearchEnv(
            flower=flower,
            canvas=canvas,
            max_steps=trajectory.num_points + 5,
        )

    env.flower_center_xy = np.array(
        [0.5 * env.canvas.width, 0.5 * env.canvas.height],
        dtype=np.float32,
    )
    env.canvas_image = env.get_canvas_image()

    actions = deora_actions_for_env(
        trajectory,
        env,
        scale_mode=scale_mode,
        head_standoff=head_standoff,
        depth=depth,
    )
    tip_xy = np.array([env._denormalize_canvas_xy(action[2:4]) for action in actions])
    head_xy = np.array([env._denormalize_canvas_xy(action[:2]) for action in actions])

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    else:
        fig = ax.figure

    ax.imshow(
        env.canvas_image,
        cmap="gray",
        origin="upper",
        extent=(0.0, env.canvas.width, env.canvas.height, 0.0),
    )
    ax.plot(tip_xy[:, 0], tip_xy[:, 1], color=line_color, linewidth=1.5, label="proboscis tip")

    ax.scatter(tip_xy[0, 0], tip_xy[0, 1], color="lime", s=36, zorder=3, label="start")
    ax.scatter(tip_xy[-1, 0], tip_xy[-1, 1], color="orange", s=36, zorder=3, label="end")
    ax.scatter(
        [env.flower_center_xy[0]],
        [env.flower_center_xy[1]],
        marker="x",
        color="black",
        s=60,
        linewidths=1.8,
        label="flower center",
    )

    ax.set_title(
        f"{trajectory.metadata.name} {trajectory.metadata.visit} "
        f"(c={trajectory.metadata.c_value:g}, n={trajectory.num_points})"
    )
    ax.set_xlabel("canvas x")
    ax.set_ylabel("canvas y")
    ax.set_xlim(0.0, env.canvas.width)
    ax.set_ylim(env.canvas.height, 0.0)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="upper right", frameon=True)
    fig.tight_layout()
    return fig, ax, trajectory, env

In [ ]:
example_track = REPO_ROOT / "data/deora_et_al/ProboscisTracking_dataFolders/v3/FilteredTracks/FinalCompiledTracks/FirstVisit/c-1_m3_FirstVisit.csv"

fig, ax, trajectory, env = plot_deora_trajectory_on_env(
    example_track,
    max_radius_normalized=None,
    depth=1.0,
)
plt.show()